<a href="https://colab.research.google.com/github/ved-ilpate/Leaf_detection/blob/main/proj_testing_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/My Drive'))

['dataset', 'Colab Notebooks']


In [3]:
dataset_path = '/content/drive/My Drive/dataset/Testing'
print (os.listdir(dataset_path))

['Apple___Apple_scab', 'Corn_(maize)___healthy', 'Apple___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Non_leaf']


In [4]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from google.colab import files

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale = 1./255,
    validation_split = 0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    subset = 'training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    subset = 'validation'
)

print("Class:", train_data.class_indices)

Found 8584 images belonging to 5 classes.
Found 2143 images belonging to 5 classes.
Class: {'Apple___Apple_scab': 0, 'Apple___healthy': 1, 'Corn_(maize)___Northern_Leaf_Blight': 2, 'Corn_(maize)___healthy': 3, 'Non_leaf': 4}


In [6]:
base_model = MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = 'imagenet' # imagenet = model train on 14+ millon images (Transfer learning)
)

base_model.trainable = False #freezes pretrained layers (which means Do NOT change existing learned weights)

model = models.Sequential([  # layer by layer processing
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = 'relu'), # 128 neurons, each neuron learn different pattern & relu remove weak signals
    layers.Dropout(0.3),
    layers.Dense(5, activation = 'softmax'),  # softmax = convert o/p into probability (confidence)
])

model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_data,
    validation_data = val_data,
    epochs = 5
)

Epoch 1/5
 11/269 ━━━━━━━━━━━━━━━━━━━━ 31:09 7s/step - accuracy: 0.5713 - loss: 1.1785

In [ ]:
model.save('/content/drive/My Drive/dataset/Testing/leaf_disease_model.keras')

In [ ]:
from google.colab import files

class_names = list(train_data.class_indices.keys())

def predict_image(img_path):

    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array)[0]

    print("\nRaw predictions:")
    for cls, prob in zip(class_names, predictions):
        print(f"{cls}: {prob*100:.2f}%")

    predicted_index = np.argmax(predictions)
    predicted_class = class_names[predicted_index]

    confidence = round(float(np.max(predictions)) * 100, 2)

    if predicted_class == "Non_leaf":
        print("❌ Not a leaf")
        print("Please upload a plant leaf image.")
    else:
        print("✅ Leaf Detected")
        print(f"Disease/Category : {predicted_class}")
        print(f"Confidence        : {confidence}%")

    plt.imshow(img)
    plt.title(f"{predicted_class} ({confidence}%)")
    plt.axis('off')
    plt.show()


uploaded = files.upload()

img_path = list(uploaded.keys())[0]

predict_image(img_path)